# 8.2 - Document Processing Pipeline

**Module 8 - RAG Systems** | Netsetos GenAI Engineering

Before retrieval can work, raw documents have to become clean, retrievable chunks. This notebook builds that ingestion stage in pure Python: a multi-format loader, three chunking strategies, a harness that measures which one retrieves best, and a production-shaped pipeline class - then adds the two upgrades (contextual chunking, content-hash dedup) that separate demo code from deployable code.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

One install line covers the whole lesson: the Gemini SDK for embeddings, PyMuPDF for PDF text, BeautifulSoup for HTML, MarkItDown for the other office formats, plus numpy for cosine math and python-dotenv for key loading. It is commented out because it only needs to run once on a fresh machine (Colab or a clean venv).

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai numpy pymupdf pymupdf4llm beautifulsoup4 markitdown python-dotenv -q  # noqa


**Explanation**

Environment prep, not logic - it installs every third-party package the later cells import. Nothing here touches a document or an API.

**How the code works - step by step**
- **`google-genai`** - the Gemini client used for embeddings (`gemini-embedding-001`) and the contextual-chunking LLM call.
- **`numpy`** - array math for cosine similarity in the evaluation harness.
- **`pymupdf`** / **`pymupdf4llm`** - PDF text and Markdown extraction (the tier-1 parser).
- **`beautifulsoup4`** - strips scripts/nav/footer out of HTML.
- **`markitdown`** - one converter for DOCX/XLSX/HTML → Markdown.
- **`python-dotenv`** - reads keys from a `.env` file so they never get hardcoded.

**In one line:** install everything the notebook imports, once.

**What you'll see:** (no output - environment prep)

## Setup - load your API key

The embedding cells and the contextual-chunking cell all call Gemini, which reads its credential from the environment. This cell seeds `GOOGLE_API_KEY` from the environment (or a `.env` file) without ever writing the key into the notebook.

> **Needs a Google AI Studio key** (set `GOOGLE_API_KEY` from aistudio.google.com).

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Still environment prep, not a model call - it just makes sure a key variable exists before any client is constructed. `setdefault` only fills the value if it is not already set, so an already-exported key wins.

**How the code works - step by step**
- **`import os`** - access to the process environment.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - guarantees the variable exists (empty by default) so `genai.Client()` can pick it up; the comment points you to aistudio.google.com to get one.

**In one line:** make sure the API key lives in the environment, not in the code.

**What you'll see:** (no output - environment prep)

## 1 - Build a document loader for any format

Before you can chunk anything you need text, and every format hides it differently - PDFs bury it in positioned glyphs, HTML wraps it in navigation and scripts, Markdown decorates it with syntax. This cell builds one `DocumentLoader` class whose `load()` dispatches on file extension and returns the same contract for all four formats: clean plain text plus a little metadata.

In [ ]:
# DOCUMENT LOADERS - Extract text from any format
# pip install pymupdf beautifulsoup4
import re
from pathlib import Path

class DocumentLoader:
    """Load and extract text from PDF, HTML, Markdown, or plain text."""

    def load(self, path: str) -> dict:
        ext = Path(path).suffix.lower()
        loaders = {".pdf": self._pdf, ".html": self._html,
                   ".md": self._markdown, ".txt": self._text}
        loader = loaders.get(ext, self._text)
        text = loader(path)
        text = re.sub(r'[ \t]+', ' ', text)   # collapse spaces ONLY
        text = re.sub(r'\n{3,}', '\n\n', text).strip()  # keep paragraph breaks for the recursive chunker
        return {"source": path, "text": text, "words": len(text.split()), "format": ext}

    def _pdf(self, path):
        import pymupdf  # "import fitz" is the legacy alias
        doc = pymupdf.open(path)
        return " ".join(page.get_text() for page in doc)

    def _html(self, path):
        from bs4 import BeautifulSoup
        with open(path, encoding="utf-8") as f: soup = BeautifulSoup(f.read(), "html.parser")
        [s.decompose() for s in soup(["script","style","nav","footer"])]
        return soup.get_text(" ")

    def _markdown(self, path):
        with open(path, encoding="utf-8") as f: text = f.read()
        text = re.sub(r'#{1,6}\s+', '', text)  # remove headers markup
        text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)  # links
        text = re.sub(r'[*_`~]+', '', text)  # formatting
        return text

    def _text(self, path):
        with open(path, encoding="utf-8") as f: return f.read()

# Demo with simulated files
loader = DocumentLoader()
print("DocumentLoader supports: .pdf, .html, .md, .txt")
print("  PDF: pymupdf extracts text, tables, even images-as-text")
print("  HTML: BeautifulSoup strips scripts/nav/footer, keeps content")
print("  MD: regex removes #headers, [links](url), **bold**")
print("  All output: clean plaintext ready for chunking")

# Output:
#   DocumentLoader supports: .pdf, .html, .md, .txt
#   PDF: pymupdf extracts text, tables, even images-as-text
#   HTML: BeautifulSoup strips scripts/nav/footer, keeps content
#   MD: regex removes #headers, [links](url), **bold**
#   All output: clean plaintext ready for chunking

**Explanation**

A single dispatcher class, not four scripts: `load` looks at the extension, calls the matching private opener, then runs the same cleanup on whatever comes back. Read it as one input contract (text + source + word count + format) that the rest of the pipeline can rely on regardless of what walked in the door.

**How the code works - step by step**
- **`load`** - reads the suffix, looks it up in a dict of openers (unknown extensions fall back to `_text`), extracts, then collapses runs of spaces/tabs and 3+ newlines to `\n\n` - deliberately keeping paragraph breaks so the recursive chunker later can see them.
- **`_pdf`** - opens with `pymupdf` and joins `page.get_text()` across every page.
- **`_html`** - parses with BeautifulSoup, `decompose()`s script/style/nav/footer, then returns the remaining text.
- **`_markdown`** - strips `#` headers, `[link](url)` syntax, and `*_`~` formatting with regex.
- **`_text`** - just reads the file (every `open` uses `encoding="utf-8"`, which matters the moment a Devanagari byte appears).

**In one line:** dispatch on extension → open with the right technique → return one clean-text contract.

**What you'll see:** Four printed lines summarizing what each format's opener does (PDF via pymupdf, HTML via BeautifulSoup stripping, MD via regex) and that all of them output clean plaintext ready for chunking - no file is actually loaded in the demo.

## 2 - Three chunking strategies, coded and compared

Lesson 8.1 showed why chunks must exist; this cell codes the three standard ways to decide WHERE to cut - by count (fixed), by structure (recursive), by meaning (semantic) - and runs all three on the same short document so you can read the actual chunks each one produces.

In [ ]:
# THREE CHUNKING STRATEGIES - Coded and compared
import re
import numpy as np

# ── Strategy 1: Fixed-size chunks ──
def chunk_fixed(text, size=200, overlap=50):
    """Split every N words with overlap. Simple but dumb."""
    assert 0 < overlap < size, "need 0 < overlap < size"
    words = text.split()
    chunks = []
    for i in range(0, len(words), size - overlap):
        c = " ".join(words[i:i+size])
        if len(c.split()) >= 20: chunks.append(c)
        elif chunks: chunks[-1] += " " + c  # merge short tail - never drop text
    return chunks

# ── Strategy 2: Recursive (structure-aware) ──
def chunk_recursive(text, max_size=200, separators=None):
    """Split by paragraph, then sentence, then word. Respects structure."""
    separators = separators or ["\n\n", "\n", ". ", " "]
    chunks = []
    def _split(text, sep_idx=0):
        if len(text.split()) <= max_size:
            if len(text.split()) >= 15: chunks.append(text.strip())
            return
        if sep_idx >= len(separators):
            chunks.append(" ".join(text.split()[:max_size]))
            return
        parts = text.split(separators[sep_idx])
        current = ""
        for part in parts:
            if len((current + part).split()) <= max_size:
                current += separators[sep_idx] + part
            else:
                if current.strip(): _split(current.strip(), sep_idx + 1)
                current = part
        if current.strip(): _split(current.strip(), sep_idx + 1)
    _split(text)
    return chunks

# ── Strategy 3: Semantic (meaning-aware) ──
def chunk_semantic(text, threshold=0.22):
    """Group sentences by similarity. Word-overlap (Jaccard) is the cheap
    proxy here; production uses sentence embeddings + cosine (Lesson 8.1)."""
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.split())>=3]
    chunks, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        prev_words = set(" ".join(current[-2:]).lower().split())
        curr_words = set(sentences[i].lower().split())
        overlap = len(prev_words & curr_words) / max(len(prev_words | curr_words), 1)
        if overlap >= threshold:  # same topic (Jaccard cutoff)
            current.append(sentences[i])
        else:  # topic shift = new chunk
            chunks.append(" ".join(current))
            current = [sentences[i]]
    if current: chunks.append(" ".join(current))
    return chunks

# ── Compare all 3 on the same document ──
doc = """Netsetos is an educational technology company founded in 2024 in Hyderabad.
We offer courses in Generative AI, Data Science, and Cloud Computing.

Our refund policy allows full refunds within 7 days of purchase.
After 7 days, we offer a 50 percent refund up to 30 days.
After 30 days, no refunds are available.

The GenAI course costs 14999 rupees and includes lifetime access.
Students also get Discord community access and weekly live sessions.
Google Cloud credits worth 5000 rupees are included.

Live classes are held daily at 7 PM IST on YouTube.
Recordings are uploaded within 2 hours after the session.
The course completion rate is 85 percent with 4.8 star rating."""

print("Chunking Strategy Comparison:\n")
for name, fn in [("Fixed (60w)", lambda t: chunk_fixed(t,60,10)),
                  ("Recursive", lambda t: chunk_recursive(t,60)),
                  ("Semantic", lambda t: chunk_semantic(t))]:
    chunks = fn(doc)
    print(f"  {name}: {len(chunks)} chunks")
    for i,c in enumerate(chunks):
        print(f"    [{i}] ({len(c.split())}w) {c[:65]}...")
    print()

# Output: (abridged)
#   Chunking Strategy Comparison:
#   Fixed (60w): 5 chunks   [0] (60w) Netsetos is an educational technology company...
#   Recursive: 4 chunks     [0] (21w) Netsetos is an educational technology company...
#   Semantic: 7 chunks      [0] (35w) Netsetos is an educational technology company...

**Explanation**

Three independent functions plus one head-to-head bake-off on a shared FAQ string. Each function draws chunk boundaries by a different rule, and the loop at the bottom prints how many chunks each rule yields and a preview of the first one - the point is to SEE that the same text splits three different ways.

**How the code works - step by step**
- **`chunk_fixed`** - walks the word list in strides of `size - overlap`, so consecutive chunks share `overlap` words; a too-short tail (<20 words) is merged back into the previous chunk instead of dropped.
- **`chunk_recursive`** - tries separators in order (`\n\n` → `\n` → `. ` → ` `), recursing to a finer separator only when a piece is still over `max_size`; respects paragraph then sentence boundaries.
- **`chunk_semantic`** - splits into sentences, then measures Jaccard word-overlap between each sentence and the current group; overlap ≥ threshold means same topic (extend the chunk), below means topic shift (start a new one). Uses cheap word-overlap as a stand-in for the real embedding+cosine version.
- **the comparison loop** - runs `Fixed(60,10)`, `Recursive(60)`, and `Semantic` on the Netsetos FAQ `doc` and prints chunk counts + previews.

**In one line:** cut by count, by structure, or by meaning - same document, three different chunk sets.

**What you'll see:** A comparison header then one block per strategy on the same doc (abridged output shows Fixed ≈ 5 chunks, Recursive ≈ 4, Semantic ≈ 7), each line listing the chunk index, its word count, and the first ~65 characters.

## 3 - Measure which chunking retrieves best

Don't argue about chunking in the abstract - measure it. This cell writes down four real questions, embeds every chunk and every question with Gemini, ranks chunks by cosine similarity, and scores whether the top-ranked chunk actually contains the answer. That score is recall@1.

> **Needs a Google AI Studio key** - this cell makes live embedding calls.

In [ ]:
# CHUNKING EVALUATION - Which strategy retrieves best?
# pip install google-genai numpy
# Requires chunk_fixed / chunk_recursive / chunk_semantic and doc from 02_chunking_strategies.py
from google import genai
from google.genai import types
import numpy as np

client = genai.Client()  # reads GOOGLE_API_KEY from the environment
EMBED = "gemini-embedding-001"

def embed_batch(texts, task, batch=100):
    """Embed in sub-batches (the API caps texts per request) - never loop per chunk."""
    out = []
    for i in range(0, len(texts), batch):
        r = client.models.embed_content(
            model=EMBED, contents=texts[i:i+batch],
            config=types.EmbedContentConfig(task_type=task))
        out.extend(np.array(e.values) for e in r.embeddings)
    return out

def eval_retrieval(chunks, test_cases):
    """Does the correct chunk rank #1 for each question?"""
    embs = embed_batch(chunks, "RETRIEVAL_DOCUMENT")
    qs = embed_batch([q for q, _ in test_cases], "RETRIEVAL_QUERY")
    correct = 0
    for qe, (question, expected_keyword) in zip(qs, test_cases):
        scores = [(i, float(np.dot(qe, e) / (np.linalg.norm(qe) * np.linalg.norm(e))))
                  for i, e in enumerate(embs)]
        scores.sort(key=lambda x: -x[1])
        if expected_keyword.lower() in chunks[scores[0][0]].lower():
            correct += 1
    return correct / len(test_cases)

# Test questions with expected keywords in the correct chunk
test_cases = [
    ("What is the refund policy?", "refund"),
    ("How much does the course cost?", "14999"),
    ("When are live classes?", "7 PM"),
    ("What is the completion rate?", "85"),
]

print("Retrieval Accuracy by Chunking Strategy:\n")
for name, chunks in [
    ("Fixed", chunk_fixed(doc, 60, 10)),
    ("Recursive", chunk_recursive(doc, 60)),
    ("Semantic", chunk_semantic(doc)),
]:
    acc = eval_retrieval(chunks, test_cases)
    print(f"  {name:12s}: {acc*100:.0f}% " + "█" * int(acc * 20))

# Output: (your corpus WILL differ - that is exactly why you measure)
#   Retrieval Accuracy by Chunking Strategy:
#   Fixed       : 50% ██████████
#   Recursive   : 100% ████████████████████
#   Semantic    : 75% ███████████████

**Explanation**

A measurement harness, not a model of chunking. It reuses the three chunkers and `doc` from cell 2, embeds documents and queries with DIFFERENT task types, and turns the abstract question 'which strategy is best?' into a single number per strategy on this specific corpus.

**How the code works - step by step**
- **`embed_batch`** - embeds a list of texts in sub-batches of 100 (the API caps texts per request), tagging each call with a `task_type`; returns numpy vectors. Never loops one call per chunk.
- **`eval_retrieval`** - embeds the chunks as `RETRIEVAL_DOCUMENT` and the questions as `RETRIEVAL_QUERY`, computes cosine similarity of each question against every chunk, sorts descending, and counts a hit when the expected keyword appears in the rank-1 chunk; returns hits / total.
- **`test_cases`** - four (question, expected-keyword) pairs (refund, 14999, 7 PM, 85) that pin each answer to its correct chunk.
- **the scoring loop** - runs Fixed / Recursive / Semantic through `eval_retrieval` and prints an ASCII bar per strategy.

**In one line:** embed chunks as documents, questions as queries → rank by cosine → count rank-1 hits = recall@1.

**What you'll see:** A per-strategy accuracy readout with ASCII bars; the sample output shows Fixed 50%, Recursive 100%, Semantic 75% - and a comment warning your corpus WILL differ, which is exactly why you measure rather than trust a default.

## 4 - The complete production pipeline

Cells 1-3 were separate experiments; production needs one object with one contract - text in, index-ready chunk records out. This `DocumentPipeline` class wires clean → chunk → embed into a single `process()` call and emits records carrying an id, the text, its embedding, and metadata.

> **Needs a Google AI Studio key** - `process()` embeds every chunk.

In [ ]:
# COMPLETE DOCUMENT PROCESSING PIPELINE
# Requires the chunk_* functions and doc from 02_chunking_strategies.py
from google import genai
from google.genai import types
import re

client = genai.Client()
EMBED = "gemini-embedding-001"

class DocumentPipeline:
    """Load → Clean → Chunk → Embed. Ready for a vector DB."""

    def __init__(self, strategy="recursive", chunk_size=200, overlap=50):
        assert overlap < chunk_size, "overlap must be smaller than chunk_size"
        self.strategy = strategy
        self.chunk_size = chunk_size
        self.overlap = overlap

    def _clean(self, text):
        text = re.sub(r'[ \t]+', ' ', text)   # spaces only - NEVER eat \n\n,
        text = re.sub(r'\n{3,}', '\n\n', text)  # or recursive chunking goes blind
        text = re.sub(r'Page \d+ of \d+', '', text)  # strip page furniture
        text = re.sub(r'http\S+', '', text)  # strip bare URLs
        return text.strip()

    def process(self, text, source="doc"):
        text = self._clean(text)
        if self.strategy == "fixed":
            chunks = chunk_fixed(text, self.chunk_size, self.overlap)
        elif self.strategy == "recursive":
            chunks = chunk_recursive(text, self.chunk_size)
        else:
            chunks = chunk_semantic(text)

        # Batch in sub-slices - the API caps texts per request, so slice;
        # a per-chunk loop is the classic beginner mistake: slower, rate-limited
        embs = []
        for i in range(0, len(chunks), 100):
            r = client.models.embed_content(
                model=EMBED, contents=chunks[i:i+100],
                config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"))
            embs.extend(e.values for e in r.embeddings)
        return [{"id": f"{source}_{i}", "text": c,
                 "embedding": list(embs[i]),
                 "source": source, "words": len(c.split())}
                for i, c in enumerate(chunks)]

pipe = DocumentPipeline(strategy="recursive", chunk_size=80)
results = pipe.process(doc, source="netsetos_faq")
print(f"Pipeline: {len(results)} chunks, ready for vector DB")
for r in results:
    print(f"  {r['id']:20s} {r['words']:3d}w  {r['text'][:50]}...")

# Output:
#   Pipeline: 4 chunks, ready for vector DB
#   netsetos_faq_0        58w  Netsetos is an educational technology company fo...
#   netsetos_faq_1        42w  Our refund policy allows full refunds within 7 d...
#   netsetos_faq_2        45w  The GenAI course costs 14999 rupees and includes...
#   netsetos_faq_3        40w  Live classes are held daily at 7 PM IST on YouTu...

**Explanation**

An assembly-line class: `__init__` fixes the strategy and sizes, `_clean` normalizes text without destroying structure, and `process` runs the fixed sequence of stations and boxes each unit with a serial number. It reuses the chunkers from cell 2 and adds the two hygiene steps (structure-safe cleaning, batched embedding) that make it deployable.

**How the code works - step by step**
- **`__init__`** - stores `strategy`, `chunk_size`, `overlap`; asserts overlap < chunk_size.
- **`_clean`** - collapses spaces/tabs and 3+ newlines but preserves `\n\n`, and strips page furniture (`Page X of Y`) and bare URLs - never eats the paragraph breaks the recursive chunker needs.
- **`process`** - cleans, dispatches to the chosen chunker, then embeds chunks in sub-slices of 100 as `RETRIEVAL_DOCUMENT`, and returns a list of records `{id, text, embedding, source, words}` with positional ids like `netsetos_faq_0`.

**In one line:** clean (structure-safe) → chunk by strategy → batch-embed → emit indexed-ready records.

**What you'll see:** `Pipeline: 4 chunks, ready for vector DB`, then one line per record showing its id (`netsetos_faq_0` … `_3`), word count, and a 50-char text preview - the FAQ split cleanly along its paragraph topics (company, refunds, pricing, live classes).

## 5 - Contextual chunking: stamp each chunk with its origin

Recursive chunking loses context at every cut - 'the fee is 2 percent' no longer says which fee, which product, which year. Contextual chunking (Anthropic, Sep 2024) fixes that by having an LLM prepend a one-sentence situating stamp to each chunk before it is embedded.

> **Needs a Google AI Studio key** - this cell calls `gemini-2.5-flash`.

In [ ]:
# Contextual chunking in one function: stamp each chunk with its origin
def contextualize(chunk, doc_summary):
    prompt = (f"Document: {doc_summary}\n\nChunk: {chunk}\n\n"
              "Write ONE sentence situating this chunk in the document. Sentence only.")
    ctx = client.models.generate_content(model="gemini-2.5-flash",
                                         contents=prompt).text.strip()
    return ctx + " " + chunk  # prepend, THEN embed

print(contextualize("The fee is 2 percent.",
                     "Netsetos 2026 refund and fees policy"))

# Output:
#   This sentence states the processing fee in the Netsetos 2026 refund
#   and fees policy. The fee is 2 percent.

**Explanation**

A single-purpose helper wrapping one LLM call: given a chunk and a short document summary, it asks the model for one situating sentence and glues that sentence to the front of the chunk. The stamp travels with the chunk into the embedder, so the vector now encodes its own context.

**How the code works - step by step**
- **`contextualize`** - builds a prompt containing the document summary and the chunk, asks `gemini-2.5-flash` for exactly one situating sentence, strips it, and returns `context + " " + chunk` (prepend first, THEN embed).
- **the demo call** - runs it on the bare chunk `"The fee is 2 percent."` with the summary `"Netsetos 2026 refund and fees policy"`.

**In one line:** ask an LLM for a one-sentence 'where this came from', prepend it, then embed.

**What you'll see:** The original sentence with an LLM-generated situating sentence in front of it - e.g. a line stating this is the processing fee in the Netsetos 2026 refund and fees policy, followed by the original 'The fee is 2 percent.'

## 6 - Content-hash ids and a clean metadata record

The two cheapest production wins in the whole pipeline: give each chunk an id derived from its own text (so re-ingesting the same content upserts instead of duplicating), and normalize Unicode to one canonical byte form (so the same Hindi word encoded two ways still matches). This cell shows both in one small `make_record`.

In [ ]:
# The two cheapest wins in this step: content-hash ids + a real metadata record
import hashlib, unicodedata

def make_record(chunk, source, page, section):
    chunk = unicodedata.normalize("NFC", chunk)  # one canonical byte form
    cid = hashlib.sha256(chunk.encode()).hexdigest()[:16]  # same text -> same id
    return {"id": cid, "text": chunk, "source": source,
            "page": page, "section": section}

r1 = make_record("Full refund within 7 days.", "policy.pdf", 3, "refunds")
r2 = make_record("Full refund within 7 days.", "policy_v2.pdf", 1, "refunds")
print(r1["id"] == r2["id"])  # re-ingestion upserts instead of duplicating

# Output:
#   True

**Explanation**

A record-builder, not a model call. It demonstrates that a content-hash id makes ingestion idempotent - identical text always produces the same id regardless of which file it came from - which is what stops accidental double-ingestion from polluting the index.

**How the code works - step by step**
- **`make_record`** - NFC-normalizes the chunk to one canonical byte form, hashes it with SHA-256 and takes the first 16 hex chars as the id, and returns `{id, text, source, page, section}`.
- **the two demo records** - `r1` and `r2` hold the SAME sentence but come from different files/pages/sections; the print compares their ids.

**In one line:** same text → same content-hash id → re-ingestion upserts instead of duplicating.

**What you'll see:** A single line: `True` - proving that identical chunk text yields an identical id even though the two records list different source files, pages, and would otherwise be treated as separate documents.

You built the whole ingestion stage from the ground up: a loader that flattens four formats to one text contract, three chunkers you can see the actual cuts of, a recall@1 harness that turns "which chunking is best?" into a number, and a pipeline class that emits index-ready records - then bolted on contextual chunking (stamp each chunk with its origin) and content-hash ids (re-ingestion upserts instead of duplicates). The recurring lesson across every cell: cleaning must never destroy the structure the chunker depends on, embedding calls are always batched never looped, and you measure on YOUR corpus rather than trusting defaults. Next, Lesson 8.3 gives these chunks a real home in a production vector store, and Lesson 8.11 upgrades the four-question smoke test here into a full evaluation harness.